In [7]:
# Run only if the libraries are not already installed
!pip install plotly ipywidgets -q

In [8]:
import pandas as pd
from google.colab import drive

drive.mount("/content/drive")
import pandas as pd
from pathlib import Path

shared_folder = Path(
    "/content/drive/MyDrive/montreal-transit-service-analysis/tableau_exports"
)

weekday_weekend = pd.read_csv(
    shared_folder / "weekday_weekend_dashboard.csv"
)

stop_map_data = pd.read_csv(
    shared_folder / "stop_map_dashboard.csv"
)

service_comparison = pd.read_csv(
    shared_folder / "service_comparison_dashboard.csv"
)
print("weekday_weekend:", weekday_weekend.shape)
print("stop_map_data:", stop_map_data.shape)
print("service_comparison:", service_comparison.shape)
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

from plotly.subplots import make_subplots
from IPython.display import display, HTML
import ipywidgets as widgets

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
weekday_weekend: (166, 11)
stop_map_data: (8746, 6)
service_comparison: (1929, 15)


In [23]:
PLOTLY_TEMPLATE = "plotly_white"

WEEKDAY_COLOR = "#1565C0"     # Dark blue
WEEKEND_COLOR = "#F57C00"     # Strong orange
INCREASE_COLOR = "#138A36"    # Dark green
REDUCTION_COLOR = "#C62828"   # Dark red

def format_figure(fig, title=None, height=450):
    fig.update_layout(
        template=PLOTLY_TEMPLATE,
        title=title,
        height=height,
        font=dict(family="Arial", size=13),
        margin=dict(l=40, r=30, t=70, b=40),
        hoverlabel=dict(font_size=13)
    )
    return fig

In [10]:
dashboard_data = weekday_weekend.copy()

# Make route names readable
dashboard_data["route_label"] = (
    dashboard_data["route_short_name"]
    .fillna(dashboard_data["route_id"].astype(str))
    .astype(str)
)

dashboard_data["trip_difference"] = (
    dashboard_data["Weekend"] - dashboard_data["Weekday"]
)

dashboard_data["percentage_change"] = np.where(
    dashboard_data["Weekday"] > 0,
    (
        (dashboard_data["Weekend"] - dashboard_data["Weekday"])
        / dashboard_data["Weekday"]
    ) * 100,
    np.nan
)

dashboard_data["change_type"] = np.where(
    dashboard_data["trip_difference"] >= 0,
    "Weekend increase",
    "Weekend reduction"
)

In [18]:
from IPython.display import display, HTML

number_of_routes = dashboard_data["route_id"].nunique()
median_weekday = dashboard_data["Weekday"].median()
median_weekend = dashboard_data["Weekend"].median()

median_change = (
    (median_weekend - median_weekday)
    / median_weekday
) * 100

change_arrow = "▲" if median_change >= 0 else "▼"

change_color = (
    "#1E8449"
    if median_change >= 0
    else "#C0392B"
)

change_background = (
    "#E8F8F0"
    if median_change >= 0
    else "#FDEDEC"
)

display(HTML(f"""
<div style="
    display:flex;
    flex-wrap:wrap;
    gap:18px;
    margin:20px 0;
    font-family:Arial, sans-serif;
">

    <!-- Paired routes -->
    <div style="
        flex:1;
        min-width:180px;
        padding:20px;
        border-radius:12px;
        background:#F2F4F4;
        border-left:6px solid #566573;
        box-shadow:0 2px 6px rgba(0,0,0,0.10);
    ">
        <div style="
            font-size:14px;
            color:#4D5656;
            margin-bottom:6px;
        ">
            Paired routes
        </div>

        <div style="
            font-size:30px;
            font-weight:700;
            color:#273746;
        ">
            {number_of_routes:,}
        </div>
    </div>

    <!-- Median weekday trips -->
    <div style="
        flex:1;
        min-width:180px;
        padding:20px;
        border-radius:12px;
        background:#E8F1FB;
        border-left:6px solid #2F75B5;
        box-shadow:0 2px 6px rgba(0,0,0,0.10);
    ">
        <div style="
            font-size:14px;
            color:#34495E;
            margin-bottom:6px;
        ">
            Median weekday trips
        </div>

        <div style="
            font-size:30px;
            font-weight:700;
            color:#1F4E79;
        ">
            {median_weekday:,.0f}
        </div>
    </div>

    <!-- Median weekend trips -->
    <div style="
        flex:1;
        min-width:180px;
        padding:20px;
        border-radius:12px;
        background:#FFF1E6;
        border-left:6px solid #F28E2B;
        box-shadow:0 2px 6px rgba(0,0,0,0.10);
    ">
        <div style="
            font-size:14px;
            color:#6E4B2A;
            margin-bottom:6px;
        ">
            Median weekend trips
        </div>

        <div style="
            font-size:30px;
            font-weight:700;
            color:#C65D00;
        ">
            {median_weekend:,.0f}
        </div>
    </div>

    <!-- Change in median -->
    <div style="
        flex:1;
        min-width:180px;
        padding:20px;
        border-radius:12px;
        background:{change_background};
        border-left:6px solid {change_color};
        box-shadow:0 2px 6px rgba(0,0,0,0.10);
    ">
        <div style="
            font-size:14px;
            color:#4D5656;
            margin-bottom:6px;
        ">
            Change in median
        </div>

        <div style="
            font-size:30px;
            font-weight:700;
            color:{change_color};
        ">
            {change_arrow} {abs(median_change):.1f}%
        </div>
    </div>

</div>
"""))

In [28]:
# ---------------------------------------------------------
# Prepare dashboard data
# ---------------------------------------------------------

plot_data = dashboard_data.melt(
    id_vars=["route_id", "route_label"],
    value_vars=["Weekday", "Weekend"],
    var_name="Service type",
    value_name="Scheduled trips"
)

top_routes = (
    dashboard_data
    .assign(
        max_trips=dashboard_data[
            ["Weekday", "Weekend"]
        ].max(axis=1)
    )
    .nlargest(20, "max_trips")
    .sort_values("max_trips")
)

top_routes_long = top_routes.melt(
    id_vars=["route_id", "route_label"],
    value_vars=["Weekday", "Weekend"],
    var_name="Service type",
    value_name="Scheduled trips"
)

largest_changes = (
    dashboard_data
    .dropna(subset=["percentage_change"])
    .assign(
        absolute_change=lambda data: data[
            "percentage_change"
        ].abs()
    )
    .nlargest(15, "absolute_change")
    .sort_values("percentage_change")
)


# ---------------------------------------------------------
# Create dashboard layout
# ---------------------------------------------------------

fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=(
        "Distribution of Scheduled Trips",
        "Weekday–Weekend Relationship",
        "Top 20 Routes by Scheduled Trips",
        "Largest Weekend Percentage Changes"
    ),
    vertical_spacing=0.17,
    horizontal_spacing=0.14
)


# ---------------------------------------------------------
# 1. Distribution box plots
# ---------------------------------------------------------

for service_type, color in [
    ("Weekday", WEEKDAY_COLOR),
    ("Weekend", WEEKEND_COLOR)
]:
    values = plot_data.loc[
        plot_data["Service type"].eq(service_type),
        "Scheduled trips"
    ]

    fig.add_trace(
        go.Box(
            y=values,
            name=service_type,
            legendgroup=service_type,
            marker_color=color,
            line_color=color,
            boxmean=True,
            showlegend=True,
            hovertemplate=(
                f"<b>{service_type}</b><br>"
                "Scheduled trips: %{y:.0f}"
                "<extra></extra>"
            )
        ),
        row=1,
        col=1
    )


# ---------------------------------------------------------
# 2. Weekday versus weekend scatter plot
# ---------------------------------------------------------

fig.add_trace(
    go.Scatter(
        x=dashboard_data["Weekday"],
        y=dashboard_data["Weekend"],
        mode="markers",
        name="Routes",
        showlegend=False,
        text=dashboard_data["route_label"],
        customdata=dashboard_data[
            ["trip_difference", "percentage_change"]
        ].to_numpy(),



        marker={
            "color": dashboard_data["percentage_change"],
            "coloraxis": "coloraxis",
            "size": 10,
            "opacity": 0.9,
            "line": {
                "color": "#333333",
                "width": 0.8
            }
        },


        hovertemplate=(
            "<b>Route %{text}</b><br>"
            "Weekday trips: %{x:.0f}<br>"
            "Weekend trips: %{y:.0f}<br>"
            "Trip difference: %{customdata[0]:+.0f}<br>"
            "Weekend change: %{customdata[1]:+.1f}%"
            "<extra></extra>"
        )
    ),
    row=1,
    col=2
)


# Equality reference line

max_value = dashboard_data[
    ["Weekday", "Weekend"]
].max().max()

fig.add_trace(
    go.Scatter(
        x=[0, max_value],
        y=[0, max_value],
        mode="lines",
        name="Equal service",
        line={
            "color": "#7F8C8D",
            "dash": "dash",
            "width": 2
        },
        hoverinfo="skip",
        showlegend=True
    ),
    row=1,
    col=2
)


# ---------------------------------------------------------
# 3. Top 20 route bars
# ---------------------------------------------------------

for service_type, color in [
    ("Weekday", WEEKDAY_COLOR),
    ("Weekend", WEEKEND_COLOR)
]:
    subset = top_routes_long.loc[
        top_routes_long["Service type"].eq(service_type)
    ]

    fig.add_trace(
        go.Bar(
            x=subset["Scheduled trips"],
            y=subset["route_label"],
            name=service_type,
            legendgroup=service_type,
            orientation="h",
            marker_color=color,
            showlegend=False,
            hovertemplate=(
                f"<b>{service_type}</b><br>"
                "Route: %{y}<br>"
                "Scheduled trips: %{x:.0f}"
                "<extra></extra>"
            )
        ),
        row=2,
        col=1
    )


# ---------------------------------------------------------
# 4. Routes with largest percentage changes
# ---------------------------------------------------------

change_colors = np.where(
    largest_changes["percentage_change"].ge(0),
    INCREASE_COLOR,
    REDUCTION_COLOR
)

fig.add_trace(
    go.Bar(
        x=largest_changes["percentage_change"],
        y=largest_changes["route_label"],
        orientation="h",
        name="Weekend change",
        marker_color=change_colors,
        showlegend=False,
        customdata=largest_changes[
            ["Weekday", "Weekend"]
        ].to_numpy(),
        hovertemplate=(
            "<b>Route %{y}</b><br>"
            "Weekday trips: %{customdata[0]:.0f}<br>"
            "Weekend trips: %{customdata[1]:.0f}<br>"
            "Weekend change: %{x:+.1f}%"
            "<extra></extra>"
        )
    ),
    row=2,
    col=2
)


# ---------------------------------------------------------
# Dashboard formatting
# ---------------------------------------------------------

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=950,
    title={
        "text": "Transit Service Dashboard: Weekday versus Weekend",
        "x": 0.5,
        "xanchor": "center",
        "y": 0.98
    },
    barmode="group",
    font={
        "family": "Arial",
        "size": 12
    },

    # Move the categorical legend above the charts
    legend={
        "orientation": "h",
        "x": 0.5,
        "xanchor": "center",
        "y": 1.03,
        "yanchor": "bottom",
        "title_text": "",
        "bgcolor": "rgba(255,255,255,0.90)",
        "bordercolor": "#D5D8DC",
        "borderwidth": 1
    },


    coloraxis={
     "colorscale": [
         [0.00, "#E65100"],   # Large weekend reduction
         [0.25, "#F57C00"],   # Moderate reduction
         [0.49, "#FFE0B2"],   # Small reduction
         [0.50, "#F5F5F5"],   # Approximately equal
         [0.51, "#C8E6C9"],   # Small increase
         [0.75, "#43A047"],   # Moderate increase
         [1.00, "#00695C"] ],   # Large weekend increase
     "cmid": 0,
     "colorbar": {
         "title": {
             "text": "Weekend<br>change (%)",
             "side": "right"
         },
         "x": 1.02,
         "xanchor": "left",
         "y": 0.76,
         "yanchor": "middle",
         "len": 0.36,
         "thickness": 18,
         "outlinecolor": "#555555",
         "outlinewidth": 1,
         "tickfont": {"size": 10}
     }
  }
    ,

    # Extra room for the colour bar
    margin={
        "l": 70,
        "r": 150,
        "t": 130,
        "b": 60
    }
)


# ---------------------------------------------------------
# Axis titles and styling
# ---------------------------------------------------------

fig.update_yaxes(
    title_text="Scheduled trips",
    row=1,
    col=1
)

fig.update_xaxes(
    title_text="Weekday trips",
    row=1,
    col=2
)

fig.update_yaxes(
    title_text="Weekend trips",
    row=1,
    col=2
)

fig.update_xaxes(
    title_text="Scheduled trips",
    row=2,
    col=1
)

fig.update_yaxes(
    title_text="Route",
    row=2,
    col=1
)

fig.update_xaxes(
    title_text="Weekend change (%)",
    zeroline=True,
    zerolinewidth=2,
    zerolinecolor="#7F8C8D",
    row=2,
    col=2
)

fig.update_yaxes(
    title_text="Route",
    row=2,
    col=2
)

fig.show()

In [17]:
# Enable interactive widgets in Google Colab.
from google.colab import output as colab_output
colab_output.enable_custom_widget_manager()

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import ipywidgets as widgets

from IPython.display import display, clear_output

# Ensure Plotly uses the Colab renderer.
pio.renderers.default = "colab"


# ---------------------------------------------------------
# Dashboard settings
# ---------------------------------------------------------

WEEKDAY_COLOR = "#2F75B5"
WEEKEND_COLOR = "#E45756"
INCREASE_COLOR = "#59A14F"
REDUCTION_COLOR = "#E45756"
PLOTLY_TEMPLATE = "plotly_white"


# ---------------------------------------------------------
# Prepare the dashboard dataset
# ---------------------------------------------------------

dashboard_data = weekday_weekend.copy()

# Standardize the percentage-change column name.
if (
    "percentage_change" not in dashboard_data.columns
    and "weekend_change_percentage" in dashboard_data.columns
):
    dashboard_data["percentage_change"] = (
        dashboard_data["weekend_change_percentage"]
    )

# Recalculate it if neither version exists.
if "percentage_change" not in dashboard_data.columns:
    dashboard_data["percentage_change"] = (
        (
            dashboard_data["Weekend"]
            - dashboard_data["Weekday"]
        )
        / dashboard_data["Weekday"].replace(0, np.nan)
    ) * 100

# Recalculate the service ratio if necessary.
if "weekend_service_ratio" not in dashboard_data.columns:
    dashboard_data["weekend_service_ratio"] = (
        dashboard_data["Weekend"]
        / dashboard_data["Weekday"].replace(0, np.nan)
    )

# Create readable route labels if needed.
if "route_label" not in dashboard_data.columns:

    dashboard_data["route_label"] = (
        dashboard_data["route_short_name"]
        .fillna(dashboard_data["route_id"])
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
    )

    if "route_long_name" in dashboard_data.columns:
        dashboard_data["route_label"] = (
            dashboard_data["route_label"]
            + " – "
            + dashboard_data["route_long_name"]
            .fillna("Route name unavailable")
        )

# Create the change category.
dashboard_data["change_type"] = np.select(
    [
        dashboard_data["percentage_change"].gt(0),
        dashboard_data["percentage_change"].lt(0)
    ],
    [
        "Weekend increase",
        "Weekend reduction"
    ],
    default="No change"
)

# Ensure route labels are text.
dashboard_data["route_label"] = (
    dashboard_data["route_label"]
    .astype(str)
)

route_options = sorted(
    dashboard_data["route_label"]
    .dropna()
    .unique()
)


# ---------------------------------------------------------
# Create the controls
# ---------------------------------------------------------

route_selector = widgets.SelectMultiple(
    options=route_options,
    value=tuple(route_options[:5]),
    description="Routes:",
    rows=8,
    layout=widgets.Layout(
        width="450px"
    ),
    style={
        "description_width": "70px"
    }
)

metric_selector = widgets.Dropdown(
    options=[
        ("Scheduled trips", "trips"),
        ("Weekend/weekday ratio", "ratio"),
        ("Percentage change", "percentage")
    ],
    value="trips",
    description="Measure:",
    layout=widgets.Layout(
        width="300px"
    ),
    style={
        "description_width": "80px"
    }
)

chart_output = widgets.Output()


# ---------------------------------------------------------
# Dashboard-update function
# ---------------------------------------------------------

def update_route_dashboard(change=None):

    selected_routes = list(
        route_selector.value
    )

    selected_metric = (
        metric_selector.value
    )

    with chart_output:

        clear_output(wait=True)

        if not selected_routes:
            print("Select at least one route.")
            return

        filtered = dashboard_data[
            dashboard_data["route_label"]
            .isin(selected_routes)
        ].copy()

        if filtered.empty:
            print("No matching route data found.")
            return

        if selected_metric == "trips":

            chart_data = filtered.melt(
                id_vars=["route_label"],
                value_vars=[
                    "Weekday",
                    "Weekend"
                ],
                var_name="Service type",
                value_name="Scheduled trips"
            )

            fig = px.bar(
                chart_data,
                x="route_label",
                y="Scheduled trips",
                color="Service type",
                barmode="group",
                color_discrete_map={
                    "Weekday": WEEKDAY_COLOR,
                    "Weekend": WEEKEND_COLOR
                },
                labels={
                    "route_label": "Route"
                },
                title=(
                    "Scheduled Trips by Selected Route"
                )
            )

        elif selected_metric == "ratio":

            fig = px.bar(
                filtered,
                x="route_label",
                y="weekend_service_ratio",
                color="weekend_service_ratio",
                color_continuous_scale="RdYlGn",
                labels={
                    "route_label": "Route",
                    "weekend_service_ratio":
                        "Weekend/weekday ratio"
                },
                title=(
                    "Weekend Service Ratio "
                    "by Selected Route"
                )
            )

            fig.add_hline(
                y=1,
                line_dash="dash",
                line_color="gray",
                annotation_text=(
                    "Equal weekday and weekend service"
                ),
                annotation_position="top left"
            )

        else:

            fig = px.bar(
                filtered,
                x="route_label",
                y="percentage_change",
                color="change_type",
                color_discrete_map={
                    "Weekend increase":
                        INCREASE_COLOR,

                    "Weekend reduction":
                        REDUCTION_COLOR,

                    "No change":
                        "#9D9D9D"
                },
                labels={
                    "route_label": "Route",
                    "percentage_change":
                        "Weekend change (%)",
                    "change_type":
                        "Change category"
                },
                title=(
                    "Weekend Percentage Change "
                    "by Selected Route"
                )
            )

            fig.add_hline(
                y=0,
                line_dash="dash",
                line_color="gray"
            )

        fig.update_layout(
            template=PLOTLY_TEMPLATE,
            height=550,
            title={
                "x": 0.5,
                "xanchor": "center"
            },
            xaxis_title="Route",
            xaxis_tickangle=-35,
            legend_title_text="",
            margin={
                "l": 60,
                "r": 30,
                "t": 70,
                "b": 150
            }
        )

        # Explicit Colab renderer.
        fig.show(renderer="colab")


# ---------------------------------------------------------
# Connect the controls
# ---------------------------------------------------------

route_selector.observe(
    update_route_dashboard,
    names="value"
)

metric_selector.observe(
    update_route_dashboard,
    names="value"
)


# ---------------------------------------------------------
# Display the dashboard
# ---------------------------------------------------------

dashboard_controls = widgets.VBox([
    widgets.HTML(
        value=(
            "<h3>Interactive Route Explorer</h3>"
            "<p>Select one or more routes and choose "
            "the comparison measure.</p>"
        )
    ),

    widgets.HBox([
        route_selector,
        metric_selector
    ])
])

display(
    dashboard_controls,
    chart_output
)

update_route_dashboard()

Output()

In [14]:
map_data = stop_map_data.dropna(
    subset=["stop_lat", "stop_lon"]
).copy()

In [15]:
fig_map = px.scatter_map(
    map_data,
    lat="stop_lat",
    lon="stop_lon",
    color="location_category",
    hover_name="stop_name",
    hover_data={
        "stop_id": True,
        "location_category": True,
        "stop_lat": ":.5f",
        "stop_lon": ":.5f"
    },
    zoom=9,
    height=650,
    title="Transit Stops and High-Frequency Service Areas",
    color_discrete_map={
        "High-frequency line": "#E74C3C",
        "Other transit stop": "#A6A6A6"
    }
)

fig_map.update_traces(
    marker=dict(size=8, opacity=0.80)
)

fig_map.update_layout(
    map_style="carto-positron",
    template=PLOTLY_TEMPLATE,
    legend_title_text="Stop category",
    margin=dict(l=0, r=0, t=60, b=0)
)

fig_map.show()

In [16]:
from scipy.stats import wilcoxon

paired_data = dashboard_data[
    ["Weekday", "Weekend"]
].dropna()

statistic, p_value = wilcoxon(
    paired_data["Weekday"],
    paired_data["Weekend"]
)

if p_value < 0.05:
    conclusion = "Statistically significant difference"
    conclusion_color = "#27AE60"
else:
    conclusion = "No statistically significant difference"
    conclusion_color = "#E67E22"

fig_stat = go.Figure()

fig_stat.add_trace(
    go.Indicator(
        mode="number",
        value=p_value,
        number={
            "valueformat": ".3e",
            "font": {"size": 42}
        },
        title={
            "text": (
                "Wilcoxon signed-rank test<br>"
                "<span style='font-size:15px'>P-value</span>"
            )
        },
        domain={"x": [0, 0.48], "y": [0, 1]}
    )
)

fig_stat.add_trace(
    go.Indicator(
        mode="number",
        value=statistic,
        number={
            "valueformat": ".0f",
            "font": {"size": 42}
        },
        title={
            "text": (
                "Test statistic<br>"
                f"<span style='font-size:15px'>{conclusion}</span>"
            )
        },
        domain={"x": [0.52, 1], "y": [0, 1]}
    )
)

fig_stat.update_layout(
    template=PLOTLY_TEMPLATE,
    height=300,
    title=dict(
        text="Statistical Comparison of Weekday and Weekend Service",
        x=0.5
    ),
    annotations=[
        dict(
            text=(
                f"Based on {len(paired_data)} paired routes, "
                f"weekday and weekend trip levels differ significantly."
            ),
            x=0.5,
            y=-0.08,
            showarrow=False,
            font=dict(size=14, color=conclusion_color)
        )
    ]
)

fig_stat.show()

### Anomaly Detection

In [29]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
import numpy as np
import pandas as pd
import plotly.express as px

RANDOM_STATE = 42

anomaly_features = [
    "Weekday",
    "Weekend",
    "trip_difference",
    "percentage_change",
    "weekend_service_ratio"
]

anomaly_data = (
    dashboard_data
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=anomaly_features)
    .copy()
)

print("Routes included in anomaly detection:", len(anomaly_data))

Routes included in anomaly detection: 166


In [30]:
scaler = RobustScaler()

X_anomaly = scaler.fit_transform(
    anomaly_data[anomaly_features]
)

In [31]:
anomaly_model = IsolationForest(
    n_estimators=300,
    contamination=0.05,
    random_state=RANDOM_STATE
)

anomaly_data["anomaly_prediction"] = (
    anomaly_model.fit_predict(X_anomaly)
)

# Convert the model score so that larger values mean more unusual.
anomaly_data["anomaly_score"] = (
    -anomaly_model.score_samples(X_anomaly)
)

anomaly_data["anomaly_status"] = np.where(
    anomaly_data["anomaly_prediction"].eq(-1),
    "Potential anomaly",
    "Typical pattern"
)

print(
    anomaly_data["anomaly_status"]
    .value_counts()
)

anomaly_status
Typical pattern      157
Potential anomaly      9
Name: count, dtype: int64


In [32]:
anomaly_results = (
    anomaly_data[
        [
            "route_id",
            "route_label",
            "Weekday",
            "Weekend",
            "trip_difference",
            "percentage_change",
            "weekend_service_ratio",
            "anomaly_score",
            "anomaly_status"
        ]
    ]
    .sort_values("anomaly_score", ascending=False)
)

anomaly_results.head(15)

,route_id,route_label,Weekday,Weekend,trip_difference,percentage_change,weekend_service_ratio,anomaly_score,anomaly_status
61,2,2 – Ligne 2 - Orange,465.000000,313.50,-151.500000,-32.580645,0.674194,0.698344,Potential anomaly
133,55,55 – Saint-Laurent,53.600000,131.75,78.150000,145.802239,2.458022,0.690789,Potential anomaly
0,1,1 – Ligne 1 - Verte,454.000000,308.50,-145.500000,-32.048458,0.679515,0.685053,Potential anomaly
129,5,5 – Ligne 5 - Bleue,372.000000,260.00,-112.000000,-30.107527,0.698925,0.677754,Potential anomaly
32,141,141 – Jean-Talon Est,84.333333,178.75,94.416667,111.956522,2.119565,0.668852,Potential anomaly
147,747,747 – YUL Aéroport Montréal-Trudeau,291.500000,310.00,18.500000,6.346484,1.063465,0.653206,Potential anomaly
120,439,439 – Express Pie-IX,285.500000,186.75,-98.750000,-34.588441,0.654116,0.641530,Potential anomaly
117,4,4 – Ligne 4 - Jaune,314.000000,315.50,1.500000,0.477707,1.004777,0.634328,Potential anomaly
81,24,24 – Sherbrooke,201.000000,109.75,-91.250000,-45.398010,0.546020,0.610988,Potential anomaly
30,14,14 – Atateken,60.000000,26.00,-34.000000,-56.666667,0.433333,0.603489,Typical pattern


In [33]:
ANOMALY_COLOR = "#D32F2F"
TYPICAL_COLOR = "#90A4AE"

fig = px.scatter(
    anomaly_data,
    x="Weekday",
    y="Weekend",
    size="anomaly_score",
    color="anomaly_status",
    hover_name="route_label",
    hover_data={
        "Weekday": ":.0f",
        "Weekend": ":.0f",
        "trip_difference": ":+.0f",
        "percentage_change": ":+.1f",
        "weekend_service_ratio": ":.2f",
        "anomaly_score": ":.3f",
        "anomaly_status": False
    },
    color_discrete_map={
        "Potential anomaly": ANOMALY_COLOR,
        "Typical pattern": TYPICAL_COLOR
    },
    title="Unusual Weekday–Weekend Scheduled Service Patterns",
    labels={
        "Weekday": "Average weekday scheduled trips",
        "Weekend": "Average weekend scheduled trips",
        "anomaly_status": "Classification"
    }
)

max_trips = anomaly_data[
    ["Weekday", "Weekend"]
].max().max()

fig.add_shape(
    type="line",
    x0=0,
    y0=0,
    x1=max_trips,
    y1=max_trips,
    line={
        "color": "#555555",
        "dash": "dash",
        "width": 2
    }
)

fig.add_annotation(
    x=max_trips * 0.76,
    y=max_trips * 0.79,
    text="Equal weekday and weekend service",
    showarrow=False,
    font={"color": "#555555"}
)

fig.update_traces(
    marker={
        "line": {
            "color": "white",
            "width": 0.7
        }
    }
)

fig.update_layout(
    template=PLOTLY_TEMPLATE,
    height=650,
    title={
        "x": 0.5,
        "xanchor": "center"
    },
    legend_title_text="Route pattern"
)

fig.show()

In [34]:
flagged_routes = (
    anomaly_results
    .query("anomaly_status == 'Potential anomaly'")
    .reset_index(drop=True)
)

flagged_routes

,route_id,route_label,Weekday,Weekend,trip_difference,percentage_change,weekend_service_ratio,anomaly_score,anomaly_status
0,2,2 – Ligne 2 - Orange,465.000000,313.50,-151.500000,-32.580645,0.674194,0.698344,Potential anomaly
1,55,55 – Saint-Laurent,53.600000,131.75,78.150000,145.802239,2.458022,0.690789,Potential anomaly
2,1,1 – Ligne 1 - Verte,454.000000,308.50,-145.500000,-32.048458,0.679515,0.685053,Potential anomaly
3,5,5 – Ligne 5 - Bleue,372.000000,260.00,-112.000000,-30.107527,0.698925,0.677754,Potential anomaly
4,141,141 – Jean-Talon Est,84.333333,178.75,94.416667,111.956522,2.119565,0.668852,Potential anomaly
5,747,747 – YUL Aéroport Montréal-Trudeau,291.500000,310.00,18.500000,6.346484,1.063465,0.653206,Potential anomaly
6,439,439 – Express Pie-IX,285.500000,186.75,-98.750000,-34.588441,0.654116,0.641530,Potential anomaly
7,4,4 – Ligne 4 - Jaune,314.000000,315.50,1.500000,0.477707,1.004777,0.634328,Potential anomaly
8,24,24 – Sherbrooke,201.000000,109.75,-91.250000,-45.398010,0.546020,0.610988,Potential anomaly


The Isolation Forest model flags routes with unusual combinations of weekday volume, weekend volume, absolute trip difference and relative percentage change. A flagged route is not necessarily affected by a data-quality issue or service disruption; it may represent a legitimate special-purpose, leisure-oriented, seasonal or event-based route. The results therefore provide a prioritized list for further investigation rather than proof of abnormal operations